In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from bs4 import BeautifulSoup

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    StratifiedKFold,
    GridSearchCV
)

print("P05 model comparison environment ready!")

P05 model comparison environment ready!


In [3]:
TRAIN_PATH = Path("../data/processed/train.csv")

train_df = pd.read_csv(TRAIN_PATH)

X_train = train_df["review"]
y_train = train_df["sentiment"]

print("Training samples:", len(X_train))
print("\nClass distribution:")
print(y_train.value_counts())

Training samples: 40000

Class distribution:
sentiment
positive    20000
negative    20000
Name: count, dtype: int64


In [4]:
def strip_html_batch(texts):
    return [
        BeautifulSoup(text, "html.parser").get_text(separator=" ")
        for text in texts
    ]

print("HTML cleaning function ready!")

HTML cleaning function ready!


In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Five-fold stratified cross-validation ready!")

Five-fold stratified cross-validation ready!


In [6]:
# P05: Model comparison pipeline

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer

# Reusable text-classification pipeline
def strip_html_batch(texts):
    return [
        BeautifulSoup(text, "html.parser").get_text(separator=" ")
        for text in texts
    ]

pipeline = Pipeline([
    (
        "html",
        FunctionTransformer(
            strip_html_batch,
            validate=False
        )
    ),
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000
        )
    ),
    ("classifier", LogisticRegression(max_iter=1000))
])

print("Model-tuning pipeline created!")

Model-tuning pipeline created!


In [8]:
# Tune Logistic Regression using 5-fold cross-validation

lr_params = {
    "classifier__C": [0.1, 1, 10]
}

lr_search = GridSearchCV(
    estimator=pipeline,
    param_grid=lr_params,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

lr_search.fit(X_train, y_train)

print("\nBest Logistic Regression parameters:")
print(lr_search.best_params_)

print(f"Best CV accuracy: {lr_search.best_score_:.4f}")

Fitting 5 folds for each of 3 candidates, totalling 15 fits

Best Logistic Regression parameters:
{'classifier__C': 10}
Best CV accuracy: 0.9125


In [9]:
# P05: Tune Multinomial Naive Bayes

from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer

nb_pipeline = Pipeline([
    (
        "html",
        FunctionTransformer(
            strip_html_batch,
            validate=False
        )
    ),
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000
        )
    ),
    ("classifier", MultinomialNB())
])

nb_params = {
    "classifier__alpha": [0.1, 0.5, 1.0, 2.0]
}

nb_search = GridSearchCV(
    estimator=nb_pipeline,
    param_grid=nb_params,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

nb_search.fit(X_train, y_train)

print("\nBest Naive Bayes parameters:")
print(nb_search.best_params_)

print(f"Best CV accuracy: {nb_search.best_score_:.4f}")

Fitting 5 folds for each of 4 candidates, totalling 20 fits

Best Naive Bayes parameters:
{'classifier__alpha': 0.1}
Best CV accuracy: 0.8874


In [10]:
# P05: Tune Linear SVM

from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer

svm_pipeline = Pipeline([
    (
        "html",
        FunctionTransformer(
            strip_html_batch,
            validate=False
        )
    ),
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000
        )
    ),
    (
        "classifier",
        LinearSVC(random_state=42, max_iter=3000)
    )
])

svm_params = {
    "classifier__C": [0.1, 1, 10]
}

svm_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_params,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

svm_search.fit(X_train, y_train)

print("\nBest Linear SVM parameters:")
print(svm_search.best_params_)

print(f"Best CV accuracy: {svm_search.best_score_:.4f}")

Fitting 5 folds for each of 3 candidates, totalling 15 fits

Best Linear SVM parameters:
{'classifier__C': 1}
Best CV accuracy: 0.9140


In [15]:
rf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=10,
        max_features=2000,
        dtype=np.float32
    )),
    ("classifier", RandomForestClassifier(
        n_estimators=20,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

rf_search = GridSearchCV(
    rf_pipeline,
    {"classifier__max_depth": [15]},
    cv=cv,
    scoring="accuracy",
    n_jobs=1,
    verbose=1
)

rf_search.fit(X_train, y_train)

print("Best parameters:", rf_search.best_params_)
print("CV accuracy:", rf_search.best_score_)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Best parameters: {'classifier__max_depth': 15}
CV accuracy: 0.804025


In [16]:
# Consolidate cross-validation results

model_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Best Parameters": str(lr_search.best_params_),
        "CV Accuracy": lr_search.best_score_
    },
    {
        "Model": "Multinomial Naive Bayes",
        "Best Parameters": str(nb_search.best_params_),
        "CV Accuracy": nb_search.best_score_
    },
    {
        "Model": "Linear SVM",
        "Best Parameters": str(svm_search.best_params_),
        "CV Accuracy": svm_search.best_score_
    },
    {
        "Model": "Random Forest",
        "Best Parameters": str(rf_search.best_params_),
        "CV Accuracy": rf_search.best_score_
    }
])

model_results = model_results.sort_values(
    "CV Accuracy",
    ascending=False
).reset_index(drop=True)

model_results["CV Accuracy (%)"] = (
    model_results["CV Accuracy"] * 100
).round(2)

# Display results
display(model_results)

# Save results
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

model_results.to_csv(
    RESULTS_DIR / "model_comparison.csv",
    index=False
)

print("Model comparison saved successfully.")

,Model,Best Parameters,CV Accuracy,CV Accuracy (%)
0,Linear SVM,{'classifier__C': 1},0.913950,91.40
1,Logistic Regression,{'classifier__C': 10},0.912475,91.25
2,Multinomial Naive Bayes,{'classifier__alpha': 0.1},0.887400,88.74
3,Random Forest,{'classifier__max_depth': 15},0.804025,80.40


Model comparison saved successfully.
